# Part 3 — EDA: BTS Multi-Year Flights (2023–2025)

**Dataset:** ~20.6M domestic flights scraped from BTS (Jan 2023 – Dec 2025)  
**Goal:** Understand delay patterns across years, carriers, routes, and seasons before feature engineering.

In [ ]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import config

config.assert_data_exists()
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 1. Load Data

In [ ]:
# Adjust nrows=None to load the full dataset (may take several minutes)
df = config.load_bts_flights()
print(f"Shape: {df.shape}")
df.head()

## 2. Schema & Dtypes

In [ ]:
df.info(memory_usage="deep")

In [ ]:
df.describe(include="all")

## 3. Missing Values

In [ ]:
miss = df.isnull().sum()
miss[miss > 0].sort_values(ascending=False)

## 4. Target Distribution

In [ ]:
target_col = "ARR_DEL15"  # 1 = arrival delayed ≥15 min

print(df[target_col].value_counts(normalize=True).rename({0: "on-time", 1: "delayed"}))
df[target_col].value_counts().plot(kind="bar", title="Delay Distribution")
plt.show()

## 5. Delay Rate by Year

In [ ]:
year_col = "YEAR"

df.groupby(year_col)[target_col].mean().plot(
    kind="bar", title="Delay Rate by Year", ylabel="Delay Rate"
)
plt.show()

## 6. Delay Rate by Month

In [ ]:
month_col = "MONTH"

df.groupby(month_col)[target_col].mean().plot(
    kind="bar", title="Delay Rate by Month", ylabel="Delay Rate"
)
plt.xticks(rotation=0)
plt.show()

## 7. Delay Rate by Carrier

In [ ]:
carrier_col = "OP_CARRIER"

carrier_stats = (
    df.groupby(carrier_col)[target_col]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)
print(carrier_stats)
carrier_stats["mean"].plot(kind="bar", title="Delay Rate by Carrier")
plt.show()

## 8. Top Routes by Volume and Delay Rate

In [ ]:
origin_col = "ORIGIN"
dest_col   = "DEST"

df["route"] = df[origin_col] + "→" + df[dest_col]
route_stats = (
    df.groupby("route")[target_col]
    .agg(["mean", "count"])
    .query("count > 1000")
    .sort_values("mean", ascending=False)
)
print("Top 20 most-delayed routes (min 1000 flights):")
print(route_stats.head(20))

## 9. Departure Time Distribution

In [ ]:
# CRS_DEP_TIME is typically HHMM format
dep_time_col = "CRS_DEP_TIME"

df[dep_time_col].hist(bins=48, figsize=(12, 4))
plt.title("Scheduled Departure Time Distribution")
plt.xlabel("HHMM")
plt.show()

## 10. Summary

TODO: Fill in key findings after running cells above.

- Dataset shape:
- Delay rate overall:
- Peak delay months:
- Most-delayed carrier:
- Key missing value columns: